**Cell Guide**
- Cell 1: Load the merged dataset, define participants/phases/outcomes, create transformed variables, and screen outcome coverage.
- Cell 2: Check design balance before modelling and justify LMM instead of repeated-measures ANOVA.
- Cell 3: Define helper functions for LMM fitting, contrasts, H2 Spearman, FDR correction, eta-squared, and residual diagnostics.
- Cell 4: Fit H1/H3/H4 LMMs for Conditions 1-3 and H2 Spearman concordance for Condition 1 only.
- Cell 5: Check post-fit residual assumptions, model warnings, output validity, row balance, and save diagnostic plots.
- Cell 6: Create paper-ready result tables and sensitivity summaries.
- Cell 7: Save final results and diagnostics CSV files.

**Statistical Notes**
- **Model choice:** balance is checked before modelling. If participant-phase row counts are unequal, LMM is used instead of repeated-measures ANOVA because LMM can handle unbalanced repeated observations.
- **Diagnostic rule:** model choice is decided first from the repeated-measures design and balance check; residual normality and residual-spread diagnostics are then checked after fitting because they are properties of model residuals.
- **Assumptions:** after fitting, residual diagnostics are checked for residual normality and residual spread. The residual-spread check is an informal screening flag, not a formal Breusch-Pagan test; residual plots are saved for manual review (Nobre & Singer, 2007).
- **Multiple testing:** p-values are reported with Benjamini-Hochberg False Discovery Rate correction (`p_value_FDR`) to control expected false discoveries across each hypothesis family (Benjamini & Hochberg, 1995).
- **Effect size:** `eta_squared` = SS_effect / SS_total. Because the primary model is LMM rather than balanced ANOVA, SS_effect is computed from full-vs-reduced fixed-effect fitted values.
- **Boundary variance warning:** this means one random-effect variance was estimated close to zero; the fixed-effect result can still be shown, but it is interpreted with a model-diagnostic caveat.

In [1]:
# 1) Setup, data load, outcome screening, and data-scope QC
from pathlib import Path
from statistics import NormalDist
import warnings

import numpy as np, pandas as pd, matplotlib.pyplot as plt, statsmodels.formula.api as smf
from scipy.stats import skew, spearmanr, shapiro
from IPython.display import display
from statsmodels.tools.sm_exceptions import ConvergenceWarning

LOCAL_ROOT = Path(r"C:/Users/pandya/Documents/Github/docker/Paper3_Github")
ROOT = LOCAL_ROOT if LOCAL_ROOT.exists() else Path("/mnt/data")
BASE = ROOT / "output" if LOCAL_ROOT.exists() else ROOT
OUT = ROOT / "Paper3_files" / "output" if LOCAL_ROOT.exists() else ROOT / "clean_notebook_output"
OUT.mkdir(parents=True, exist_ok=True)
RAW, RESULTS, DIAGNOSTICS = BASE / "merged_all_11participants.csv", OUT / "Paper3_LMM_results_H1_H2_H3_H4.csv", OUT / "Paper3_LMM_model_diagnostics.csv"
PLOT_DIR = OUT / "Paper3_LMM_diagnostic_plots"
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 180); pd.set_option("display.max_colwidth", 120)

PARTICIPANT_ORDER = [f"P{x}" for x in [4, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]]
PHASE_ORDER = ["BikeU", "WalkU", "BikeG", "WalkG"]
PRIMARY_LMM_OUTCOMES = ["empatica_heart_rate", "empatica_sqrt_eda_tonic", "empatica_skin_temperature", "eyetracker_pupil_diameter_avg", "eyetracker_fixation_rate", "eyetracker_gaze_velocity", "eyetracker_saccade_peak_velocity"]
DESCRIPTIVE_ONLY_OUTCOMES = ["empatica_log_hrv_td_rmssd"]
OUTCOME_ORDER = PRIMARY_LMM_OUTCOMES.copy()
QUESTIONNAIRE_ORDER = ["questionnaire_current_comfort_1_7", "questionnaire_thermal_comfort_1_5", "questionnaire_environmental_comfort_mean_1_5", "questionnaire_stai6_mean_1_5", "questionnaire_prs11_total_mean_1_5"]
PREFERRED = {"Thermal physiology": "empatica_skin_temperature", "Electrodermal/autonomic arousal": "empatica_sqrt_eda_tonic", "Cardiovascular": "empatica_heart_rate", "Pupil/autonomic visual response": "eyetracker_pupil_diameter_avg", "Fixation/visual attention": "eyetracker_fixation_rate", "Saccade/oculomotor response": "eyetracker_saccade_peak_velocity", "Gaze distribution/movement": "eyetracker_gaze_velocity"}
TRANSFORM = {"empatica_sqrt_eda_tonic": "sqrt tonic EDA", "empatica_log_hrv_td_rmssd": "log RMSSD"}

print("Analysis engine : Python statsmodels MixedLM inside this notebook")
print("p_value_FDR: Benjamini-Hochberg False Discovery Rate correction within each hypothesis family")
print("eta_squared: model-based SS_effect / SS_total from full-vs-reduced fixed-effect fitted values")
print(f"Raw merged file : {RAW}\nResults file    : {RESULTS}\nDiagnostics file: {DIAGNOSTICS}")

def zscore(s):
    s = pd.to_numeric(s, errors="coerce"); sd = s.std(skipna=True)
    return (s - s.mean(skipna=True)) / sd if pd.notna(sd) and sd else np.nan

def prepare_all_rows(path=RAW):
    df = pd.read_csv(path, low_memory=False)
    df["ParticipantID"] = df["ParticipantID"].astype(str).str.strip()
    df["ParticipantID"] = df["ParticipantID"].where(df["ParticipantID"].str.startswith("P"), "P" + df["ParticipantID"])
    df["PhaseID"] = df["PhaseID"].astype(str).str.replace("_", "", regex=False).str.strip()
    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
    if "empatica_eda_tonic" in df:
        eda = pd.to_numeric(df["empatica_eda_tonic"], errors="coerce")
        df["empatica_sqrt_eda_tonic"] = np.sqrt(eda.where(eda >= 0))
    if "empatica_hrv_td_rmssd" in df:
        rmssd = pd.to_numeric(df["empatica_hrv_td_rmssd"], errors="coerce")
        df["empatica_log_hrv_td_rmssd"] = np.log(rmssd.where(rmssd > 0))
    return df

def prepare_route_rows(df):
    d = df[df["ParticipantID"].isin(PARTICIPANT_ORDER) & df["PhaseID"].isin(PHASE_ORDER)].copy()
    d["Mode"] = pd.Categorical(np.where(d["PhaseID"].isin(["BikeU", "BikeG"]), "Bike", "Walk"), categories=["Bike", "Walk"])
    d["Environment"] = pd.Categorical(np.where(d["PhaseID"].isin(["BikeU", "WalkU"]), "Urban", "Green"), categories=["Urban", "Green"])
    d["PhaseBlock"] = d["ParticipantID"] + "_" + d["PhaseID"]
    d = d.sort_values(["PhaseBlock", "Datetime"])
    d["time_in_phase_sec"] = d.groupby("PhaseBlock")["Datetime"].transform(lambda s: (s - s.min()).dt.total_seconds())
    d["time_in_phase_z"], d["DayID"] = zscore(d["time_in_phase_sec"]), d["Datetime"].dt.date
    for raw_col, stem in [("ucm_UTCI", "UTCI"), ("ucm_SND_dBA", "SND")]:
        d[f"{stem}_day_mean"] = d.groupby(["ParticipantID", "DayID"])[raw_col].transform("mean")
        d[f"{stem}_within_day"] = d[raw_col] - d[f"{stem}_day_mean"]
        d[f"{stem}_day_mean_z"], d[f"{stem}_within_day_z"] = zscore(d[f"{stem}_day_mean"]), zscore(d[f"{stem}_within_day"])
    return d

def trim_same_duration_from_start(d):
    phase_max = d.groupby(["ParticipantID", "PhaseID"], observed=True)["time_in_phase_sec"].max().reset_index(name="phase_max_sec")
    complete = phase_max.groupby("ParticipantID").filter(lambda g: set(PHASE_ORDER).issubset(set(g["PhaseID"])))
    out = d.merge(complete.groupby("ParticipantID", observed=True)["phase_max_sec"].min().reset_index(name="cut_sec"), on="ParticipantID", how="inner").query("time_in_phase_sec <= cut_sec").drop(columns="cut_sec")
    check = out.groupby(["ParticipantID", "PhaseID"], observed=True).size().unstack(fill_value=0).reindex(columns=PHASE_ORDER)
    bad = check[check.nunique(axis=1) != 1]
    if len(bad):
        print("WARNING: Condition 2 is not balanced for these participants:"); print(bad)
    return out

def subtract_baseline_mean(d, all_rows, phase):
    available = [c for c in OUTCOME_ORDER if c in all_rows.columns]
    baseline = all_rows[all_rows["ParticipantID"].isin(PARTICIPANT_ORDER) & all_rows["PhaseID"].eq(phase)].groupby("ParticipantID")[available].mean(numeric_only=True).add_suffix(f"_{phase}")
    out = d.join(baseline, on="ParticipantID")
    for col in available:
        base = f"{col}_{phase}"
        if base in baseline and baseline[base].notna().any():
            out[col] = out[col] - out[base]
    return out.drop(columns=[c for c in out.columns if c.endswith(f"_{phase}")])

all_rows = prepare_all_rows()
route_as_is = prepare_route_rows(all_rows)
route_same_start = trim_same_duration_from_start(route_as_is)
CONDITION_FRAMES = {
    "Condition 1": route_as_is,
    "Condition 2": route_same_start,
    "Condition 3": subtract_baseline_mean(route_same_start, all_rows, "Tram"),
}

def outcome_family(outcome):
    x = outcome.lower()
    tests = [("skin_temperature", "Thermal physiology"), (("eda", "scl"), "Electrodermal/autonomic arousal"), (("heart_rate", "hrv", "rmssd"), "Cardiovascular"), ("pupil", "Pupil/autonomic visual response"), ("fixation", "Fixation/visual attention"), ("saccade", "Saccade/oculomotor response"), (("gaze", "distance_from_center"), "Gaze distribution/movement"), (("acc", "vector_magnitude", "__met", "speed"), "Movement/activity covariates")]
    for keys, fam in tests:
        if any(k in x for k in ((keys,) if isinstance(keys, str) else keys)):
            return fam
    return "Other sensor outcome"

def candidate_columns(df):
    banned, helper = ("ucm_", "atmo_lys_", "questionnaire_"), ("segment_", "valid_rri", "removed_pct", "max_gap", "valid_span", "_available", "_z", "gps", "phaseblock")
    candidate_names = []
    for col in df.select_dtypes(include="number").columns:
        lc = col.lower()
        if col not in {"empatica_eda_tonic", "empatica_hrv_td_rmssd"} and not lc.startswith(banned) and not any(h in lc for h in helper) and (col.startswith("empatica") or col.startswith("eyetracker_")):
            candidate_names.append(col)
    return list(dict.fromkeys(candidate_names + ["empatica_sqrt_eda_tonic", "empatica_log_hrv_td_rmssd"]))

def screen_role(row):
    if row.Coverage == "fail" or row.near_zero_variance: return "excluded"
    if row.Outcome in PRIMARY_LMM_OUTCOMES: return "primary_lmm"
    if row.Outcome in DESCRIPTIVE_ONLY_OUTCOMES: return "descriptive_caveat"
    return "descriptive"

screen_rows = []
for col in candidate_columns(route_as_is):
    x = pd.to_numeric(route_as_is[col], errors="coerce"); n = int(x.notna().sum()); missing = 100 * (1 - n / len(route_as_is))
    near_zero_variance = x.dropna().nunique() <= 1 or (pd.notna(x.std()) and x.std() < 1e-8)
    participants, phaseblocks = int(route_as_is.loc[x.notna(), "ParticipantID"].nunique()), int(route_as_is.loc[x.notna(), "PhaseBlock"].nunique())
    coverage = "fail" if n == 0 or missing > 80 or participants < 6 or phaseblocks < 12 or near_zero_variance else "caution" if missing > 40 or participants < 8 or phaseblocks < 24 else "pass"
    fam = outcome_family(col)
    screen_rows.append({"Outcome": col, "Source": "Empatica" if col.startswith("empatica") else "Eyetracker", "Outcome_family": fam, "Preferred_representative": "yes" if PREFERRED.get(fam) == col else "no", "Transformation": TRANSFORM.get(col, "none"), "Coverage": coverage, "missing_percent": round(missing, 1), "n_rows_nonmissing": n, "n_participants": participants, "n_phaseblocks": phaseblocks, "mean": x.mean(), "sd": x.std(), "min": x.min(), "max": x.max(), "skewness": skew(x.dropna()) if n > 2 else np.nan, "near_zero_variance": near_zero_variance})

outcome_screen = pd.DataFrame(screen_rows).assign(corr_with_family_preferred=np.nan)
for fam, group in outcome_screen.groupby("Outcome_family"):
    pref = PREFERRED.get(fam)
    if pref in set(group["Outcome"]):
        base = pd.to_numeric(route_as_is[pref], errors="coerce")
        for outcome in group["Outcome"]:
            vals = pd.concat([base, pd.to_numeric(route_as_is[outcome], errors="coerce")], axis=1).dropna()
            outcome_screen.loc[outcome_screen.Outcome.eq(outcome), "corr_with_family_preferred"] = vals.corr().iloc[0, 1] if len(vals) > 20 else np.nan
outcome_screen["Role"] = outcome_screen.apply(screen_role, axis=1)
outcome_screen["Role_reason"] = np.select([outcome_screen["Role"].eq("primary_lmm"), outcome_screen["Role"].eq("descriptive_caveat"), outcome_screen["Role"].eq("excluded")], ["selected sensor-family representative for primary LMM", "descriptive/caveat only; not used for primary LMM inference", "coverage/variance or non-outcome failure"], default="screened descriptive/non-preferred variable")
outcome_screen = outcome_screen.sort_values(["Source", "Outcome_family", "Role", "Outcome"]).reset_index(drop=True)

def phase_counts(frame):
    return frame.groupby(["ParticipantID", "PhaseID"], observed=True).size().unstack(fill_value=0).reindex(index=PARTICIPANT_ORDER, columns=PHASE_ORDER)

coverage_primary = outcome_screen[outcome_screen["Outcome"].isin(OUTCOME_ORDER)][["Outcome", "Role", "Coverage", "missing_percent", "n_participants", "n_phaseblocks", "Transformation"]]
for title, obj in [("Primary LMM outcomes", coverage_primary), ("Condition 1: as-is", phase_counts(route_as_is)), ("Condition 2: same duration", phase_counts(route_same_start))]:
    print(title); display(obj)
print(f"Route rows: Condition 1={len(route_as_is)}, Condition 2={len(route_same_start)}, Condition 3={len(CONDITION_FRAMES['Condition 3'])}")
print(f"Primary LMM outcomes: {len(OUTCOME_ORDER)}")

Analysis engine : Python statsmodels MixedLM inside this notebook
p_value_FDR: Benjamini-Hochberg False Discovery Rate correction within each hypothesis family
eta_squared: model-based SS_effect / SS_total from full-vs-reduced fixed-effect fitted values
Raw merged file : C:\Users\pandya\Documents\Github\docker\Paper3_Github\output\merged_all_11participants.csv
Results file    : C:\Users\pandya\Documents\Github\docker\Paper3_Github\Paper3_files\output\Paper3_LMM_results_H1_H2_H3_H4.csv
Diagnostics file: C:\Users\pandya\Documents\Github\docker\Paper3_Github\Paper3_files\output\Paper3_LMM_model_diagnostics.csv
Primary LMM outcomes


,Outcome,Role,Coverage,missing_percent,n_participants,n_phaseblocks,Transformation
9,empatica_heart_rate,primary_lmm,pass,0.0,11,44,none
12,empatica_sqrt_eda_tonic,primary_lmm,pass,9.8,11,44,sqrt tonic EDA
17,empatica_skin_temperature,primary_lmm,pass,0.1,11,44,none
20,eyetracker_fixation_rate,primary_lmm,pass,9.5,10,40,none
24,eyetracker_gaze_velocity,primary_lmm,pass,9.5,10,40,none
30,eyetracker_pupil_diameter_avg,primary_lmm,pass,9.5,10,40,none
34,eyetracker_saccade_peak_velocity,primary_lmm,pass,9.6,10,40,none


Condition 1: as-is


PhaseID,BikeU,WalkU,BikeG,WalkG
ParticipantID,,,,
P4,129,198,59,85
P8,188,168,88,100
P9,137,199,51,82
P10,151,171,51,80
P11,118,175,63,88
P12,114,165,51,89
P13,128,177,49,90
P14,114,198,62,84
P15,110,189,54,97


Condition 2: same duration


PhaseID,BikeU,WalkU,BikeG,WalkG
ParticipantID,,,,
P4,59,59,59,59
P8,88,88,88,88
P9,51,51,51,51
P10,51,51,51,51
P11,63,63,63,63
P12,51,51,51,51
P13,49,49,49,49
P14,62,62,62,62
P15,54,54,54,54


Route rows: Condition 1=5037, Condition 2=2628, Condition 3=2628
Primary LMM outcomes: 7


In [2]:
# 2) Design balance and model-choice check before modelling

def balance_check(frame):
    counts = phase_counts(frame).fillna(0).astype(int)
    return pd.Series({
        "participant_phase_cells": int(counts.size),
        "min_rows_per_cell": int(counts.min().min()),
        "max_rows_per_cell": int(counts.max().max()),
        "equal_rows_all_participant_phase_cells": bool(counts.stack().nunique() == 1),
        "participants_equal_rows_across_their_four_phases": int((counts.nunique(axis=1) == 1).sum()),
        "participants_total": int(len(counts)),
    })

balance_decision = pd.DataFrame({name: balance_check(frame) for name, frame in CONDITION_FRAMES.items()}).T
print("Design balance check")
display(balance_decision)
print("Model-choice conclusion")
print("Condition 1 is unbalanced across participant-phase cells. Conditions 2 and 3 equalize route-phase row counts within each participant, but row counts still differ between participants because matched duration is participant-specific. Therefore the primary analysis uses LMM rather than repeated-measures ANOVA; the same LMM framework is kept for sensitivity conditions. Residual normality and residual-spread diagnostics are checked after fitting because they are properties of model residuals, not raw data.")


Design balance check


,participant_phase_cells,min_rows_per_cell,max_rows_per_cell,equal_rows_all_participant_phase_cells,participants_equal_rows_across_their_four_phases,participants_total
Condition 1,44,49,199,False,0,11
Condition 2,44,49,88,False,11,11
Condition 3,44,49,88,False,11,11


Model-choice conclusion
Condition 1 is unbalanced across participant-phase cells. Conditions 2 and 3 equalize route-phase row counts within each participant, but row counts still differ between participants because matched duration is participant-specific. Therefore the primary analysis uses LMM rather than repeated-measures ANOVA; the same LMM framework is kept for sensitivity conditions. Residual normality and residual-spread diagnostics are checked after fitting because they are properties of model residuals, not raw data.


In [3]:
# 3) Helper functions: LMM fitting, contrasts, H2 Spearman, false discovery rate correction, and assumption diagnostics

def fit_mixedlm(data, outcome, block, formula_terms=None):
    h4_covariates = ["UTCI_within_day_z", "SND_within_day_z", "UTCI_day_mean_z", "SND_day_mean_z"]
    formula_terms = formula_terms or ("Mode * Environment + time_in_phase_z" if block == "H1H3" else "Mode * Environment + UTCI_within_day_z * SND_within_day_z + UTCI_day_mean_z + SND_day_mean_z + time_in_phase_z")
    needed = [outcome, "Mode", "Environment", "time_in_phase_z", "ParticipantID", "PhaseBlock"] + ([] if block == "H1H3" else h4_covariates)
    model_data = data[needed].dropna().copy()
    if len(model_data) < 80 or model_data["ParticipantID"].nunique() < 8:
        return None, model_data, "insufficient data", np.nan

    def try_fit(method):
        with warnings.catch_warnings(record=True) as ws:
            warnings.simplefilter("always")
            try:
                fit = smf.mixedlm(f"{outcome} ~ {formula_terms}", model_data, groups=model_data["ParticipantID"], re_formula="1", vc_formula={"PhaseBlock": "0 + C(PhaseBlock)"}).fit(reml=False, method=method, maxiter=300, disp=False)
            except Exception as exc:
                return None, f"fit error ({method}): {exc}"
        msg = "; ".join(dict.fromkeys(str(w.message) for w in ws if issubclass(w.category, (ConvergenceWarning, RuntimeWarning, UserWarning))))
        return fit, msg

    fit, msg = try_fit("bfgs")
    if fit is None or (msg and any(x in msg.lower() for x in ["failed to converge", "hessian", "gradient optimization failed"])):
        fit2, msg2 = try_fit("powell")
        if fit2 is not None:
            fit, msg = fit2, (msg + "; retried powell: " + msg2) if msg else msg2
    if fit is None:
        return None, model_data, msg, np.nan
    try:
        pb_var = float(np.asarray(fit.vcomp).ravel()[0]) if getattr(fit, "vcomp", None) is not None else np.nan
    except Exception:
        pb_var = np.nan
    return fit, model_data, msg, pb_var

def estimate_se_contrast(fit, weights):
    L = np.array([weights.get(name, 0.0) for name in fit.params.index], dtype=float)
    est = float(L @ fit.params.values); se = float(np.sqrt(L @ fit.cov_params().values @ L))
    estimate_se_ratio = est / se if se > 0 else np.nan
    p_value = 2 * (1 - NormalDist().cdf(abs(estimate_se_ratio))) if np.isfinite(estimate_se_ratio) else np.nan
    return est, se, estimate_se_ratio, p_value, est - 1.96 * se, est + 1.96 * se

def eta_squared_model(data, outcome, block, reduced_terms):
    full_terms = "Mode * Environment + time_in_phase_z" if block == "H1H3" else "Mode * Environment + UTCI_within_day_z * SND_within_day_z + UTCI_day_mean_z + SND_day_mean_z + time_in_phase_z"
    full_fit, model_data, _, _ = fit_mixedlm(data, outcome, block, full_terms)
    reduced_fit, _, _, _ = fit_mixedlm(data, outcome, block, reduced_terms)
    if full_fit is None or reduced_fit is None or len(model_data) == 0:
        return np.nan
    full_fixed = np.asarray(full_fit.model.exog @ full_fit.fe_params)
    reduced_fixed = np.asarray(reduced_fit.model.exog @ reduced_fit.fe_params)
    y = np.asarray(full_fit.model.endog, dtype=float)
    ss_effect = float(np.nansum((full_fixed - reduced_fixed) ** 2))
    ss_total = float(np.nansum((y - np.nanmean(y)) ** 2))
    return ss_effect / ss_total if ss_total > 0 else np.nan

def result_row(condition, hypothesis, outcome, term, estimate, se, estimate_se_ratio, p_value, ci_low, ci_high, resid_sd, eta_squared):
    effect_sd = estimate / resid_sd if pd.notna(resid_sd) and resid_sd and np.isfinite(resid_sd) else np.nan
    return {"Condition": condition, "Hypothesis": hypothesis, "Outcome": outcome, "Term": term, "Estimate": estimate, "SE": se, "Estimate/SE": estimate_se_ratio, "CI_low": ci_low, "CI_high": ci_high, "Std_effect_residSD": effect_sd, "eta_squared": eta_squared, "p_value": p_value}

def run_condition_lmms(data, condition):
    rows, diagnostic_rows, cache = [], [], {}
    for outcome in [c for c in OUTCOME_ORDER if c in data.columns]:
        eta_h1 = eta_squared_model(data, outcome, "H1H3", "Mode + time_in_phase_z")
        eta_h3 = eta_squared_model(data, outcome, "H1H3", "Environment + time_in_phase_z")
        eta_h4 = eta_squared_model(data, outcome, "H4", "Mode * Environment + UTCI_within_day_z + SND_within_day_z + UTCI_day_mean_z + SND_day_mean_z + time_in_phase_z")
        for block in ["H1H3", "H4"]:
            fit, model_data, warn, pb_var = fit_mixedlm(data, outcome, block)
            if fit is None:
                resid_sd = resid_fit_cor = shapiro_p = residual_spread_flag = np.nan
            else:
                fixed_fit = np.asarray(fit.model.exog @ fit.fe_params)
                resid = np.asarray(fit.model.endog - fixed_fit)
                resid_sd = float(np.std(resid, ddof=1))
                resid_fit_cor = float(pd.Series(np.abs(resid)).corr(pd.Series(fixed_fit)))
                finite_resid = resid[np.isfinite(resid)]
                sample = finite_resid if len(finite_resid) <= 5000 else finite_resid[np.linspace(0, len(finite_resid) - 1, 5000).astype(int)]
                shapiro_p = float(shapiro(sample).pvalue) if 3 <= len(sample) <= 5000 else np.nan
                residual_spread_flag = abs(resid_fit_cor) > 0.20 if pd.notna(resid_fit_cor) else np.nan
            assumption_note = "check residual plots"
            if pd.notna(shapiro_p) and shapiro_p < 0.01:
                assumption_note = "residual normality caution"
            if residual_spread_flag is True:
                assumption_note = "residual spread caution"
            diagnostic_rows.append({"Condition": condition, "Analysis": block, "Outcome": outcome, "Status": "ok" if fit is not None else "failed", "n_rows": len(model_data), "n_participants": model_data["ParticipantID"].nunique() if len(model_data) else 0, "PhaseBlock_var": pb_var, "warning": str(warn or ""), "residual_sd": resid_sd, "abs_resid_fitted_cor": resid_fit_cor, "shapiro_residual_p_value": shapiro_p, "residual_spread_flag": residual_spread_flag, "assumption_note": assumption_note})
            if fit is None:
                continue
            cache[(condition, block, outcome)] = (fit, model_data)
            if block == "H1H3":
                rows.append(result_row(condition, "H1", outcome, "H1_Green_minus_Urban", *estimate_se_contrast(fit, {"Environment[T.Green]": 1, "Mode[T.Walk]:Environment[T.Green]": 0.5}), resid_sd, eta_h1))
                rows.append(result_row(condition, "H3", outcome, "H3_Walk_minus_Bike", *estimate_se_contrast(fit, {"Mode[T.Walk]": 1, "Mode[T.Walk]:Environment[T.Green]": 0.5}), resid_sd, eta_h3))
            else:
                term = "UTCI_within_day_z:SND_within_day_z"
                est, se = fit.params.get(term, np.nan), fit.bse.get(term, np.nan)
                estimate_se_ratio = est / se if pd.notna(se) and se > 0 else np.nan
                p_value = 2 * (1 - NormalDist().cdf(abs(estimate_se_ratio))) if np.isfinite(estimate_se_ratio) else np.nan
                rows.append(result_row(condition, "H4", outcome, "H4_UTCI_within_day_x_SND_within_day", est, se, estimate_se_ratio, p_value, est - 1.96 * se, est + 1.96 * se, resid_sd, eta_h4))
    return pd.DataFrame(rows), pd.DataFrame(diagnostic_rows), cache

def run_h2_concordance(route_data):
    selected_columns = [c for c in OUTCOME_ORDER + QUESTIONNAIRE_ORDER if c in route_data.columns]
    phase_mean = route_data.groupby(["ParticipantID", "PhaseID"], observed=True)[selected_columns].mean(numeric_only=True).reset_index()
    rows = []
    for outcome in [c for c in OUTCOME_ORDER if c in phase_mean.columns]:
        for qcol in [c for c in QUESTIONNAIRE_ORDER if c in phase_mean.columns]:
            d = phase_mean[[outcome, qcol]].dropna(); n = len(d)
            if n >= 8:
                rho, p_value = spearmanr(d[outcome], d[qcol])
                se = np.sqrt(1 / (n - 3)) if n > 3 and abs(rho) < 1.0 else np.nan
                ci_low, ci_high = (np.tanh(np.arctanh(rho) - 1.96 * se), np.tanh(np.arctanh(rho) + 1.96 * se)) if pd.notna(se) else (np.nan, np.nan)
            else:
                rho = p_value = se = ci_low = ci_high = np.nan
            rows.append({"Condition": "Condition 1", "Hypothesis": "H2", "Outcome": outcome, "Term": f"H2_spearman_vs_{qcol}", "Questionnaire": qcol, "Estimate": rho, "SE": se, "Estimate/SE": np.nan, "CI_low": ci_low, "CI_high": ci_high, "n_pairs": n, "eta_squared": np.nan, "p_value": p_value})
    return pd.DataFrame(rows)

def benjamini_hochberg(p_values):
    p = pd.to_numeric(pd.Series(p_values), errors="coerce")
    out = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna().sort_values()
    if valid.empty:
        return out
    n = len(valid); adjusted = valid * n / np.arange(1, n + 1)
    out.loc[valid.index] = np.minimum.accumulate(adjusted.iloc[::-1]).iloc[::-1].clip(upper=1).values
    return out

print("Helper functions loaded.")


Helper functions loaded.


In [4]:
# 4) Run H1/H3/H4 LMMs and H2 Spearman
parts, diagnostic_parts, model_cache = [], [], {}
for condition, frame in CONDITION_FRAMES.items():
    res_part, diagnostic_part, cache_part = run_condition_lmms(frame, condition)
    parts.append(res_part)
    diagnostic_parts.append(diagnostic_part)
    model_cache.update(cache_part)

results = pd.concat(parts + [run_h2_concordance(route_as_is)], ignore_index=True)
diagnostics = pd.concat(diagnostic_parts, ignore_index=True)

results["p_value_FDR"] = np.nan
for _, idx in results.groupby(["Condition", "Hypothesis"], dropna=False).groups.items():
    results.loc[idx, "p_value_FDR"] = benjamini_hochberg(results.loc[idx, "p_value"])
results["Result_from_p_value"] = np.where(results["p_value"] < 0.05, "statistically significant", "not statistically significant")
results["Result_after_FDR"] = np.where(results["p_value_FDR"] < 0.05, "statistically significant after Benjamini-Hochberg correction", "not statistically significant after Benjamini-Hochberg correction")
results["Analysis"] = np.select([results["Hypothesis"].isin(["H1", "H3"]), results["Hypothesis"].eq("H4")], ["H1H3", "H4"], default="H2")

diagnostics["warning"] = diagnostics["warning"].fillna("").astype(str)
diagnostics["diagnostic_note"] = np.select(
    [diagnostics["Status"].ne("ok"), diagnostics["warning"].str.contains("failed to converge|Gradient", case=False, regex=True), diagnostics["warning"].str.contains("boundary", case=False, regex=False), diagnostics["assumption_note"].str.contains("caution", case=False, regex=False)],
    ["model failed", "estimated after optimizer retry", "estimated with boundary variance warning", "estimated with residual-assumption caution"],
    default="estimated without captured warning",
)

notes = diagnostics[["Condition", "Analysis", "Outcome", "diagnostic_note"]]
results = results.merge(notes, on=["Condition", "Analysis", "Outcome"], how="left")
results["diagnostic_note"] = results["diagnostic_note"].fillna("Condition 1 phase-mean Spearman correlation; no mixed model fitted")
results["primary_interpretation"] = np.where(results["diagnostic_note"].str.contains("caveat|failed|caution"), "no - diagnostic caveat", "yes")

warning_table = diagnostics.loc[diagnostics["warning"].ne(""), ["Condition", "Analysis", "Outcome", "warning"]]
assumption_cautions = diagnostics.loc[diagnostics["diagnostic_note"].str.contains("caution", case=False, regex=False), ["Condition", "Analysis", "Outcome", "shapiro_residual_p_value", "abs_resid_fitted_cor", "assumption_note"]]
run_summary = pd.DataFrame({"metric": ["result_rows", "lmm_diagnostic_rows", "primary_lmm_outcomes", "failed_lmm_fits", "warning_rows", "assumption_caution_rows", "results_file", "diagnostics_file"], "value": [len(results), len(diagnostics), results.loc[results["Hypothesis"].isin(["H1", "H3", "H4"]), "Outcome"].nunique(), int(diagnostics["Status"].ne("ok").sum()), int((diagnostics["warning"] != "").sum()), len(assumption_cautions), RESULTS.name, DIAGNOSTICS.name]})
print("Run summary"); display(run_summary)
print("Captured model warnings"); display(warning_table if len(warning_table) else pd.DataFrame({"Message": ["No captured model warnings."]}))
print("Residual assumption cautions"); display(assumption_cautions if len(assumption_cautions) else pd.DataFrame({"Message": ["No residual assumption cautions from automated checks."]}))


Run summary


,metric,value
0,result_rows,98
1,lmm_diagnostic_rows,42
2,primary_lmm_outcomes,7
3,failed_lmm_fits,0
4,warning_rows,1
5,assumption_caution_rows,41
6,results_file,Paper3_LMM_results_H1_H2_H3_H4.csv
7,diagnostics_file,Paper3_LMM_model_diagnostics.csv


Captured model warnings


,Condition,Analysis,Outcome,warning
1,Condition 1,H4,empatica_heart_rate,The MLE may be on the boundary of the parameter space.


Residual assumption cautions


,Condition,Analysis,Outcome,shapiro_residual_p_value,abs_resid_fitted_cor,assumption_note
0,Condition 1,H1H3,empatica_heart_rate,1.124789e-20,0.007926,residual normality caution
2,Condition 1,H1H3,empatica_sqrt_eda_tonic,1.681558e-44,0.088620,residual normality caution
3,Condition 1,H4,empatica_sqrt_eda_tonic,3.015235e-19,0.281868,residual spread caution
4,Condition 1,H1H3,empatica_skin_temperature,1.530916e-30,-0.039570,residual normality caution
5,Condition 1,H4,empatica_skin_temperature,4.268025e-13,0.069658,residual normality caution
6,Condition 1,H1H3,eyetracker_pupil_diameter_avg,3.079253e-28,0.211752,residual spread caution
7,Condition 1,H4,eyetracker_pupil_diameter_avg,6.245454e-24,0.296662,residual spread caution
8,Condition 1,H1H3,eyetracker_fixation_rate,1.456965e-36,-0.037155,residual normality caution
9,Condition 1,H4,eyetracker_fixation_rate,3.615213e-35,-0.094235,residual normality caution
10,Condition 1,H1H3,eyetracker_gaze_velocity,3.329026e-26,0.352637,residual spread caution


In [5]:
# 5) Model diagnostics, assumptions, validity checks, and saved plots

diagnostic_view = diagnostics[diagnostics["Outcome"].isin(OUTCOME_ORDER)].copy()
diagnostic_view["Outcome"] = pd.Categorical(diagnostic_view["Outcome"], OUTCOME_ORDER, ordered=True)
diagnostic_table = diagnostic_view.sort_values(["Condition", "Analysis", "Outcome"]).rename(columns={"Analysis": "Model_block", "Status": "Fit_status", "PhaseBlock_var": "PhaseBlock_variance", "residual_sd": "Residual_SD", "abs_resid_fitted_cor": "abs_residual_vs_fitted_cor", "shapiro_residual_p_value": "Shapiro_residual_p_value", "residual_spread_flag": "Residual_spread_flag", "assumption_note": "Assumption_note"})
issues = diagnostic_table.loc[diagnostic_table["Fit_status"].astype(str).ne("ok") | diagnostic_table["warning"].fillna("").astype(str).ne("")].copy()
assumption_table = diagnostic_table.assign(
    normality_check=lambda d: np.where(pd.to_numeric(d["Shapiro_residual_p_value"], errors="coerce") < 0.01, "caution", "no automated flag"),
    residual_spread_check=lambda d: np.where(d["Residual_spread_flag"].astype(str).str.lower().eq("true"), "caution", "no automated flag"),
    model_warning=lambda d: np.where(d["warning"].fillna("").astype(str).ne(""), "warning captured", "none"),
)[["Condition", "Model_block", "Outcome", "Shapiro_residual_p_value", "normality_check", "abs_residual_vs_fitted_cor", "residual_spread_check", "model_warning", "Assumption_note", "diagnostic_note"]]
assumption_summary = assumption_table.groupby(["Condition", "Model_block"], observed=True).agg(
    models=("Outcome", "count"),
    normality_cautions=("normality_check", lambda s: int((s == "caution").sum())),
    residual_spread_cautions=("residual_spread_check", lambda s: int((s == "caution").sum())),
    captured_warnings=("model_warning", lambda s: int((s == "warning captured").sum())),
).reset_index()
range_checks = pd.DataFrame({"check": ["p_value range", "p_value_FDR range", "SE positive for LMM rows", "CI ordering", "finite eta_squared for LMM rows"], "passed": [results["p_value"].dropna().between(0, 1).all(), results["p_value_FDR"].dropna().between(0, 1).all(), (results.loc[results["Hypothesis"].isin(["H1", "H3", "H4"]), "SE"].dropna() > 0).all(), (results.loc[results["CI_low"].notna() & results["CI_high"].notna(), "CI_low"] <= results.loc[results["CI_low"].notna() & results["CI_high"].notna(), "CI_high"]).all(), np.isfinite(results.loc[results["Hypothesis"].isin(["H1", "H3", "H4"]), "eta_squared"].dropna()).all()]})
balance_summary = pd.DataFrame([{ "Condition": c, **{f"{p}_rows": int(phase_counts(frame).get(p, pd.Series(dtype=int)).sum()) for p in PHASE_ORDER}} for c, frame in CONDITION_FRAMES.items()])

normality_cautions = int(assumption_summary["normality_cautions"].sum())
residual_spread_cautions = int(assumption_summary["residual_spread_cautions"].sum())
warning_count = int(assumption_summary["captured_warnings"].sum())
assumption_conclusion = (
    f"Automated residual checks flagged {normality_cautions} residual-normality cautions, "
    f"{residual_spread_cautions} residual-spread cautions, and {warning_count} captured model warning(s). "
    "Therefore LMM remains the correct model choice for the unbalanced repeated-measures design, but individual model estimates should be interpreted with the diagnostic caveats shown below and the saved residual plots."
)

print("Warnings or failed fits"); display(issues if len(issues) else pd.DataFrame({"Message": ["No failed fits or captured warnings."]}))
if issues["warning"].fillna("").str.contains("boundary", case=False, regex=False).any():
    print("Boundary variance warning note: a random-effect variance was estimated close to zero; keep the fixed-effect result visible, but interpret it with a model-diagnostic caveat.")
print("Residual assumption summary"); display(assumption_summary)
print("Diagnostic conclusion"); print(assumption_conclusion)
print("Per-model assumption table"); display(assumption_table.astype(object).fillna("not applicable"))
print("Result-column validity checks"); display(range_checks)
print("Condition row-balance summary"); display(balance_summary)

PLOT_DIR.mkdir(parents=True, exist_ok=True)
for old_plot in PLOT_DIR.glob("condition_*.png"):
    old_plot.unlink()
def plot_fit_diagnostic(condition, block, outcome):
    key = (condition, block, outcome)
    if key not in model_cache:
        return None
    from scipy import stats
    fit, _ = model_cache[key]
    fitted = np.asarray(fit.model.exog @ fit.fe_params)
    resid = np.asarray(fit.model.endog - fitted)
    std_resid = (resid - np.nanmean(resid)) / np.nanstd(resid)
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    axes = axes.ravel()
    axes[0].scatter(fitted, std_resid, s=6, alpha=0.25); axes[0].axhline(0, color="black", lw=0.8)
    axes[0].set_title("standardized residual vs fitted"); axes[0].set_xlabel("fixed-effect fitted"); axes[0].set_ylabel("standardized residual")
    stats.probplot(std_resid[np.isfinite(std_resid)], dist="norm", plot=axes[1]); axes[1].set_title("Q-Q plot")
    axes[2].hist(fitted[np.isfinite(fitted)], bins=30); axes[2].set_title("fitted distribution")
    axes[3].hist(std_resid[np.isfinite(std_resid)], bins=30); axes[3].set_title("residual distribution")
    fig.suptitle(f"{block} | {outcome} | {condition}"); plt.tight_layout()
    path = PLOT_DIR / f"{condition.lower().replace(' ', '_')}_{block}_{outcome}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)
    return path

plot_paths = [plot_fit_diagnostic(c, b, o) for c, b, o in sorted(model_cache.keys())]
plot_paths = [p for p in plot_paths if p is not None]
print(f"Saved plots: {len(plot_paths)} PNG files")
print(f"Plot folder: {PLOT_DIR}")


Warnings or failed fits


,Condition,Model_block,Outcome,Fit_status,n_rows,n_participants,PhaseBlock_variance,warning,Residual_SD,abs_residual_vs_fitted_cor,Shapiro_residual_p_value,Residual_spread_flag,Assumption_note,diagnostic_note
1,Condition 1,H4,empatica_heart_rate,ok,5029,11,29.238403,The MLE may be on the boundary of the parameter space.,9.451265,0.076791,1.394251e-19,False,residual normality caution,estimated with boundary variance warning


Boundary variance warning note: a random-effect variance was estimated close to zero; keep the fixed-effect result visible, but interpret it with a model-diagnostic caveat.
Residual assumption summary


,Condition,Model_block,models,normality_cautions,residual_spread_cautions,captured_warnings
0,Condition 1,H1H3,7,7,2,0
1,Condition 1,H4,7,7,3,1
2,Condition 2,H1H3,7,7,2,0
3,Condition 2,H4,7,7,2,0
4,Condition 3,H1H3,7,7,2,0
5,Condition 3,H4,7,7,3,0


Diagnostic conclusion
Automated residual checks flagged 42 residual-normality cautions, 14 residual-spread cautions, and 1 captured model warning(s). Therefore LMM remains the correct model choice for the unbalanced repeated-measures design, but individual model estimates should be interpreted with the diagnostic caveats shown below and the saved residual plots.
Per-model assumption table


,Condition,Model_block,Outcome,Shapiro_residual_p_value,normality_check,abs_residual_vs_fitted_cor,residual_spread_check,model_warning,Assumption_note,diagnostic_note
0,Condition 1,H1H3,empatica_heart_rate,1.124789e-20,caution,0.007926,no automated flag,none,residual normality caution,estimated with residual-assumption caution
2,Condition 1,H1H3,empatica_sqrt_eda_tonic,1.681558e-44,caution,0.088620,no automated flag,none,residual normality caution,estimated with residual-assumption caution
4,Condition 1,H1H3,empatica_skin_temperature,1.530916e-30,caution,-0.039570,no automated flag,none,residual normality caution,estimated with residual-assumption caution
6,Condition 1,H1H3,eyetracker_pupil_diameter_avg,3.079253e-28,caution,0.211752,caution,none,residual spread caution,estimated with residual-assumption caution
8,Condition 1,H1H3,eyetracker_fixation_rate,1.456965e-36,caution,-0.037155,no automated flag,none,residual normality caution,estimated with residual-assumption caution
10,Condition 1,H1H3,eyetracker_gaze_velocity,3.329026e-26,caution,0.352637,caution,none,residual spread caution,estimated with residual-assumption caution
12,Condition 1,H1H3,eyetracker_saccade_peak_velocity,1.310346e-35,caution,0.132139,no automated flag,none,residual normality caution,estimated with residual-assumption caution
1,Condition 1,H4,empatica_heart_rate,1.394251e-19,caution,0.076791,no automated flag,warning captured,residual normality caution,estimated with boundary variance warning
3,Condition 1,H4,empatica_sqrt_eda_tonic,3.015235e-19,caution,0.281868,caution,none,residual spread caution,estimated with residual-assumption caution
5,Condition 1,H4,empatica_skin_temperature,4.268025e-13,caution,0.069658,no automated flag,none,residual normality caution,estimated with residual-assumption caution


Result-column validity checks


,check,passed
0,p_value range,True
1,p_value_FDR range,True
2,SE positive for LMM rows,True
3,CI ordering,True
4,finite eta_squared for LMM rows,True


Condition row-balance summary


,Condition,BikeU_rows,WalkU_rows,BikeG_rows,WalkG_rows
0,Condition 1,1425,1996,657,959
1,Condition 2,657,657,657,657
2,Condition 3,657,657,657,657


Saved plots: 42 PNG files
Plot folder: C:\Users\pandya\Documents\Github\docker\Paper3_Github\Paper3_files\output\Paper3_LMM_diagnostic_plots


In [6]:
# 6) Paper-ready result objects and compact sensitivity/p-value checks

result_view = results.loc[results["Condition"].eq("Condition 1") & results["Outcome"].isin(OUTCOME_ORDER)].copy()
result_view["Outcome"] = pd.Categorical(result_view["Outcome"], OUTCOME_ORDER, ordered=True)

def clean_lmm_table(hypothesis, term, question):
    result_columns = ["Outcome", "Term", "Estimate", "SE", "Estimate/SE", "CI_low", "CI_high", "eta_squared", "p_value", "p_value_FDR", "Result_after_FDR", "primary_interpretation", "diagnostic_note"]
    out = result_view.loc[result_view["Hypothesis"].eq(hypothesis) & result_view["Term"].eq(term), result_columns].copy()
    out.insert(0, "Hypothesis", hypothesis); out.insert(1, "Question", question)
    return out.sort_values("Outcome").reset_index(drop=True)

h1_table = clean_lmm_table("H1", "H1_Green_minus_Urban", "Green minus Urban")
h3_table = clean_lmm_table("H3", "H3_Walk_minus_Bike", "Walk minus Bike")
h4_table = clean_lmm_table("H4", "H4_UTCI_within_day_x_SND_within_day", "Within-day heat x noise")
h2_table = result_view.loc[result_view["Hypothesis"].eq("H2"), ["Outcome", "Term", "Questionnaire", "Estimate", "SE", "CI_low", "CI_high", "p_value", "p_value_FDR", "Result_after_FDR"]].copy().sort_values(["Outcome", "Questionnaire"]).reset_index(drop=True)
sensitivity_summary = results.loc[results["Outcome"].isin(OUTCOME_ORDER), ["Condition", "Hypothesis", "Outcome", "Term", "Estimate", "SE", "Estimate/SE", "eta_squared", "p_value", "p_value_FDR", "CI_low", "CI_high", "Result_after_FDR", "primary_interpretation", "diagnostic_note"]].copy()
conclusion_counts = results[results["Outcome"].isin(OUTCOME_ORDER)].assign(significant_raw_p=lambda d: d["p_value"].lt(0.05), significant_after_FDR=lambda d: d["p_value_FDR"].lt(0.05)).groupby(["Condition", "Hypothesis"], observed=True)[["significant_raw_p", "significant_after_FDR"]].sum().reset_index()

print("Inline result tables show primary Condition 1 only. Conditions 2 and 3 are sensitivity checks and are included in the saved CSV and sensitivity summary.")
print("H2 uses phase-mean Spearman correlations for Condition 1 only, not LMM.")
print("Condition 1 H1 table"); display(h1_table.round(4).astype(object).fillna("not applicable"))
print("Condition 1 H3 table"); display(h3_table.round(4).astype(object).fillna("not applicable"))
print("Condition 1 H4 table"); display(h4_table.round(4).astype(object).fillna("not applicable"))
print("Condition 1 H2 Spearman table"); display(h2_table.round(4).astype(object).fillna("not applicable"))
print("Raw and Benjamini-Hochberg false-discovery-rate corrected result counts"); display(conclusion_counts)
print("Sensitivity summary preview"); display(sensitivity_summary.head().round(4).astype(object).fillna("not applicable"))


Inline result tables show primary Condition 1 only. Conditions 2 and 3 are sensitivity checks and are included in the saved CSV and sensitivity summary.
H2 uses phase-mean Spearman correlations for Condition 1 only, not LMM.
Condition 1 H1 table


,Hypothesis,Question,Outcome,Term,Estimate,SE,Estimate/SE,CI_low,CI_high,eta_squared,p_value,p_value_FDR,Result_after_FDR,primary_interpretation,diagnostic_note
0,H1,Green minus Urban,empatica_heart_rate,H1_Green_minus_Urban,1.4512,1.6608,0.8738,-1.8039,4.7063,0.0138,0.3822,0.5351,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
1,H1,Green minus Urban,empatica_sqrt_eda_tonic,H1_Green_minus_Urban,-0.3565,0.1365,-2.6125,-0.6240,-0.0890,0.0153,0.0090,0.0210,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
2,H1,Green minus Urban,empatica_skin_temperature,H1_Green_minus_Urban,-1.3583,0.1754,-7.7441,-1.7020,-1.0145,0.0827,0.0000,0.0000,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
3,H1,Green minus Urban,eyetracker_pupil_diameter_avg,H1_Green_minus_Urban,0.4979,0.0763,6.5254,0.3484,0.6475,0.2385,0.0000,0.0000,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
4,H1,Green minus Urban,eyetracker_fixation_rate,H1_Green_minus_Urban,-0.0321,0.0616,-0.5222,-0.1528,0.0885,0.0018,0.6015,0.7018,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
5,H1,Green minus Urban,eyetracker_gaze_velocity,H1_Green_minus_Urban,-28.3102,90.7814,-0.3118,-206.2418,149.6214,0.1471,0.7552,0.7552,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
6,H1,Green minus Urban,eyetracker_saccade_peak_velocity,H1_Green_minus_Urban,-197.4377,191.6513,-1.0302,-573.0743,178.1989,0.0175,0.3029,0.5301,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution


Condition 1 H3 table


,Hypothesis,Question,Outcome,Term,Estimate,SE,Estimate/SE,CI_low,CI_high,eta_squared,p_value,p_value_FDR,Result_after_FDR,primary_interpretation,diagnostic_note
0,H3,Walk minus Bike,empatica_heart_rate,H3_Walk_minus_Bike,0.2182,1.6580,0.1316,-3.0315,3.4678,0.0057,0.8953,0.8953,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
1,H3,Walk minus Bike,empatica_sqrt_eda_tonic,H3_Walk_minus_Bike,0.3671,0.1364,2.6916,0.0998,0.6344,0.0170,0.0071,0.0124,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
2,H3,Walk minus Bike,empatica_skin_temperature,H3_Walk_minus_Bike,0.9862,0.1751,5.6306,0.6429,1.3295,0.0462,0.0000,0.0000,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
3,H3,Walk minus Bike,eyetracker_pupil_diameter_avg,H3_Walk_minus_Bike,-0.1061,0.0762,-1.3918,-0.2554,0.0433,0.0281,0.1640,0.1913,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
4,H3,Walk minus Bike,eyetracker_fixation_rate,H3_Walk_minus_Bike,0.2357,0.0614,3.8399,0.1154,0.3560,0.0674,0.0001,0.0003,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
5,H3,Walk minus Bike,eyetracker_gaze_velocity,H3_Walk_minus_Bike,354.5145,90.7729,3.9055,176.5996,532.4294,0.3971,0.0001,0.0003,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
6,H3,Walk minus Bike,eyetracker_saccade_peak_velocity,H3_Walk_minus_Bike,291.3804,191.5932,1.5208,-84.1422,666.9031,0.0318,0.1283,0.1796,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution


Condition 1 H4 table


,Hypothesis,Question,Outcome,Term,Estimate,SE,Estimate/SE,CI_low,CI_high,eta_squared,p_value,p_value_FDR,Result_after_FDR,primary_interpretation,diagnostic_note
0,H4,Within-day heat x noise,empatica_heart_rate,H4_UTCI_within_day_x_SND_within_day,-0.2530,0.1223,-2.0692,-0.4927,-0.0134,0.0006,0.0385,0.0674,not statistically significant after Benjamini-Hochberg correction,yes,estimated with boundary variance warning
1,H4,Within-day heat x noise,empatica_sqrt_eda_tonic,H4_UTCI_within_day_x_SND_within_day,-0.0066,0.0055,-1.2069,-0.0174,0.0041,0.0000,0.2275,0.3185,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
2,H4,Within-day heat x noise,empatica_skin_temperature,H4_UTCI_within_day_x_SND_within_day,0.0525,0.0111,4.7211,0.0307,0.0743,0.0004,0.0000,0.0000,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
3,H4,Within-day heat x noise,eyetracker_pupil_diameter_avg,H4_UTCI_within_day_x_SND_within_day,0.0162,0.0044,3.6824,0.0076,0.0248,0.0008,0.0002,0.0005,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
4,H4,Within-day heat x noise,eyetracker_fixation_rate,H4_UTCI_within_day_x_SND_within_day,-0.0054,0.0061,-0.8711,-0.0174,0.0067,0.0001,0.3837,0.4476,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
5,H4,Within-day heat x noise,eyetracker_gaze_velocity,H4_UTCI_within_day_x_SND_within_day,-0.8883,1.5824,-0.5614,-3.9898,2.2132,0.0000,0.5745,0.5745,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
6,H4,Within-day heat x noise,eyetracker_saccade_peak_velocity,H4_UTCI_within_day_x_SND_within_day,-30.1992,5.9726,-5.0563,-41.9054,-18.4929,0.0006,0.0000,0.0000,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution


... (2 outputs left out)

Raw and Benjamini-Hochberg false-discovery-rate corrected result counts


,Condition,Hypothesis,significant_raw_p,significant_after_FDR
0,Condition 1,H1,3,3
1,Condition 1,H2,4,0
2,Condition 1,H3,4,4
3,Condition 1,H4,4,3
4,Condition 2,H1,3,3
5,Condition 2,H3,5,3
6,Condition 2,H4,5,5
7,Condition 3,H1,3,3
8,Condition 3,H3,5,3
9,Condition 3,H4,5,5


Sensitivity summary preview


,Condition,Hypothesis,Outcome,Term,Estimate,SE,Estimate/SE,eta_squared,p_value,p_value_FDR,CI_low,CI_high,Result_after_FDR,primary_interpretation,diagnostic_note
0,Condition 1,H1,empatica_heart_rate,H1_Green_minus_Urban,1.4512,1.6608,0.8738,0.0138,0.3822,0.5351,-1.8039,4.7063,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
1,Condition 1,H3,empatica_heart_rate,H3_Walk_minus_Bike,0.2182,1.6580,0.1316,0.0057,0.8953,0.8953,-3.0315,3.4678,not statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
2,Condition 1,H4,empatica_heart_rate,H4_UTCI_within_day_x_SND_within_day,-0.2530,0.1223,-2.0692,0.0006,0.0385,0.0674,-0.4927,-0.0134,not statistically significant after Benjamini-Hochberg correction,yes,estimated with boundary variance warning
3,Condition 1,H1,empatica_sqrt_eda_tonic,H1_Green_minus_Urban,-0.3565,0.1365,-2.6125,0.0153,0.0090,0.0210,-0.6240,-0.0890,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution
4,Condition 1,H3,empatica_sqrt_eda_tonic,H3_Walk_minus_Bike,0.3671,0.1364,2.6916,0.0170,0.0071,0.0124,0.0998,0.6344,statistically significant after Benjamini-Hochberg correction,no - diagnostic caveat,estimated with residual-assumption caution


In [7]:
# 7) Save final CSV outputs

results.to_csv(RESULTS, index=False)
print(f"Saved CSV: {RESULTS.name}")
diagnostic_table.to_csv(DIAGNOSTICS, index=False)
print(f"Saved CSV: {DIAGNOSTICS.name}")

Saved CSV: Paper3_LMM_results_H1_H2_H3_H4.csv
Saved CSV: Paper3_LMM_model_diagnostics.csv
